# Урок 3.4. Защита от галлюцинаций: эшелонированная оборона

**Модуль 3 «RAG: Retrieval-Augmented Generation на практике» · Курс «LLM для Python-разработчиков»**

Урок 3.3 закончился неудобной правдой: grounding-промпт — это **уговор, а не гарантия**. Модель соглашалась отвечать «только по контексту» — и всё равно выдавала «56 дней» вместо «до 90», отвечала из весов на «Упал прод» и подменяла вопрос про выходные соседним про ночные вызовы. Сегодня вокруг генератора строится **эшелонированная оборона** (defense in depth): несколько дешёвых рубежей, каждый из которых ловит свой класс ошибок, — так, чтобы галлюцинация должна была пробить их все, чтобы дойти до пользователя.

> ⚠️ **Формат урока.** Все ячейки с LLM и Qdrant — guarded: они запускаются, только если Ollama с `qwen2.5:3b` и контейнер Qdrant доступны (`ollama pull qwen2.5:3b`, `docker compose up -d`, затем `python service/index_kb.py`, если коллекции нет). Числа, которые упоминаются в тексте, — из живых прогонов уроков 3.1–3.3, с точным указанием источника.

## 🎯 Цели урока

После этого урока вы сможете:

- ставить **порог релевантности** и отказываться от ответа *до* вызова LLM — и объяснить, какой класс ошибок порог не ловит принципиально;
- требовать от модели **проверяемые цитаты** — и валидировать их механически, без LLM;
- запускать **self-check** дозированно — каскадом, где дешёвые проверки идут первыми;
- превратить отказ из «ошибки» в **продуктовую фичу** с подсказкой ближайших тем;
- собрать все рубежи в единый конвейер `guarded_rag_answer()` с прозрачным вердиктом.

## План

1. Карта дыр: чему нас научили уроки 3.1–3.3
2. Рубеж 0 — порог релевантности (до генерации)
3. Рубеж 1 — промпт (краткое резюме v1/v2/v3)
4. Рубеж 2 — цитаты, которые можно проверить
5. Рубеж 3 — self-check по каскаду
6. Рубеж 4 — отказ как фича
7. Сборка: guarded_rag_answer

⏱ **Время:** ~75–90 минут.

## 1. Карта дыр: чему нас научили уроки 3.1–3.3

Оборону не строят абстрактно — её строят против задокументированных атак. У нас их четыре, все пойманы метриками урока 3.3:

| инцидент | что произошло | почему промпт не спас | какой рубеж это ловит |
|---|---|---|---|
| «56 дней» (урок 3.3, `zagranitsa`) | ретривер нашёл верный документ («до 90 дней»), модель ответила числом из ниоткуда — и то ловилось, то нет («мерцание») | модель *хотела* ответить по контексту — и исказила его | цитаты (рубеж 2), self-check (рубеж 3) |
| «Упал прод» (уроки 3.1/3.3) | ретривер промахнулся (синонимы), контекст — мусор; модель ответила советами из весов | правило «только по контексту» проиграло желанию быть полезной | порог? **нет** — увидим, почему; промпт v3 + self-check |
| подмена вопроса (урок 3.3, v2) | «переработки в выходные» → уверенный ответ про *ночные вызовы* | просьба «отвечай полно» перевесила право не знать | промпт v3 (рубеж 1), self-check с вопросом |
| хрупкий детектор отказа (урок 3.3) | модель отказалась «другими словами» — метрика не узнала отказ | детектор был завязан на точную фразу | канонический отказ + детектор-паттерны (рубеж 4) |

Общий принцип: **ни один рубеж не ловит всё, но каждый дёшев, а классы ошибок у них разные.** Конвейер целиком:

```text
вопрос
  │
  ▼
retrieve ──> [Рубеж 0: top-score < порога?] ──да──> ОТКАЗ (LLM не вызывалась!)
  │ нет
  ▼
build_prompt (v3 из урока 3.3)          <- Рубеж 1: промпт
  │
  ▼
generate ──> [модель сама отказалась?] ──да──> ОТКАЗ (+ подсказка тем, Рубеж 4)
  │ нет
  ▼
[Рубеж 2: цитаты валидны? дословная цитата найдена в контексте?] ──нет──┐
  │ да                                                                   │
  ▼                                                                      ▼
[Рубеж 3: self-check - ответ подтверждён контекстом?] ──нет──> retry 1 раз,
  │ да                                                         потом ОТКАЗ
  ▼
ОТВЕТ + источники + вердикты всех проверок (в лог - урок 3.7)
```

Обратите внимание на порядок: **дешёвое — первым**. Порог — это сравнение одного числа (бесплатно), проверка цитат — регулярка (бесплатно), self-check — вызов LLM (дорого, и он последний).

In [ ]:
import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path

import httpx

sys.path.insert(0, os.getcwd())

QDRANT_URL = os.getenv("QDRANT_URL", "http://127.0.0.1:6333")
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://127.0.0.1:11434")
MODEL = os.getenv("LLM_MODEL", "qwen2.5:3b")
COLLECTION = "vectorika_rag_tfidf"
STATE_PATH = Path("service/tfidf_state.json")


def check(label, probe):
    try:
        detail = probe()
        print(f"[ OK ] {label}: {detail}")
        return True
    except Exception as error:
        print(f"[SKIP] {label}: {type(error).__name__} - зависимые ячейки пропустятся")
        return False


QDRANT_AVAILABLE = check(
    "Qdrant", lambda: "v" + httpx.get(QDRANT_URL, timeout=3).json()["version"])
OLLAMA_AVAILABLE = check(
    "Ollama и модель",
    lambda: MODEL if MODEL in [m["name"] for m in httpx.get(
        f"{OLLAMA_URL}/api/tags", timeout=3).json()["models"]] else 1 / 0)

if QDRANT_AVAILABLE:
    from qdrant_client import QdrantClient

    from service.embedder import TfidfEmbedder

    if not STATE_PATH.exists():
        subprocess.run([sys.executable, "service/index_kb.py"], check=False)
    emb = TfidfEmbedder.load(STATE_PATH)
    qdrant = QdrantClient(url=QDRANT_URL)

    def retrieve(question: str, top_k: int = 3) -> list[dict]:
        hits = qdrant.query_points(COLLECTION, query=emb.embed_query(question),
                                   limit=top_k).points
        return [{**hit.payload, "score": hit.score} for hit in hits]


def chat(messages, num_predict=400, temperature=0.0):
    response = httpx.post(f"{OLLAMA_URL}/api/chat", json={
        "model": MODEL, "messages": messages, "stream": False,
        "options": {"temperature": temperature, "num_predict": num_predict},
    }, timeout=300)
    response.raise_for_status()
    return response.json()["message"]["content"]


golden = json.loads(Path("eval/golden.json").read_text(encoding="utf-8"))

# лучший промпт по метрикам урока 3.3 (решение 2): решение об отказе отделено
# от стиля ответа - и отказы 3/3, и полнота на месте
SYSTEM_RAG_V3 = (
    "Ты — ассистент по внутренней базе знаний компании «Векторика».\n"
    "Правила:\n"
    "1. Отвечай ТОЛЬКО на основе фрагментов из блока <context>.\n"
    "2. Сначала реши: есть ли в контексте ПРЯМОЙ ответ именно на этот вопрос "
    "(а не на похожий). Если нет — ответь ровно одной фразой: "
    "«В базе знаний нет ответа на этот вопрос» и больше ничего не пиши.\n"
    "3. Если ответ есть — дай его одним полным предложением, включая все "
    "условия, сроки и цифры из контекста по теме вопроса.\n"
    "4. В конце укажи использованные источники в виде [номер]. По-русски."
)
REFUSAL_TEXT = "В базе знаний нет ответа на этот вопрос"
print("настройка готова")

## 2. Рубеж 0 — порог релевантности: отказ до генерации

Скор ретривера у нас уже есть — Qdrant возвращает косинусную близость с каждым чанком (урок 2.4). До сих пор мы на неё смотрели, но ничего не *решали*. А ведь это самый дешёвый детектор в системе: если даже лучший чанк еле похож на вопрос — генерацию можно не начинать.

Что мы знаем о скорах из живых прогонов (все числа задокументированы в уроках 3.1 и 3.3):

- офтопик «Какая столица Франции?» — top-score **~0.04–0.05** (урок 3.1): почти ноль, чистый мусор;
- нормальные вопросы golden — top-score **от 0.149 до 0.445** (урок 3.3, таблица eval);
- канарейка «Упал прод» — top-score **0.341**... у *нерелевантного* документа `meetings` (урок 3.3).

Последняя строка — главная. Порог отлично отсекает **явный офтопик** (0.05 против 0.15 — зазор в три раза), но «Упал прод» он не поймает никогда: лексический ретривер *уверенно* нашёл не то. Скор меряет похожесть слов, а не правильность документа — «уверенный мусор» проходит любой разумный порог. Для близких ловушек («переработки в выходные» — про деньги и графики в базе много) скоры тоже будут в «нормальной» зоне — проверьте на своей машине ячейкой ниже.

Как выбирать порог: **по данным, а не из головы.** Прогоняем golden, смотрим на два распределения — top-скоры отвечаемых вопросов и ловушек — и ставим границу между ними. Каждое значение порога — это trade-off: выше порог → больше ложных отказов (отвечаемый вопрос отбит), ниже → больше пропущенного мусора. Это тот же компромисс, что у любого классификатора.

In [ ]:
if QDRANT_AVAILABLE:
    print(f"{'вопрос':<52} {'top-score':>9}  {'top-doc':<12} тип")
    print("-" * 88)
    rows = []
    for item in golden:
        found = retrieve(item["question"], top_k=1)
        top = found[0]
        kind = "отвечаемый" if item["answerable"] else "ЛОВУШКА"
        rows.append((item["question"], top["score"], top["doc_id"], kind))
        print(f"{item['question'][:50]:<52} {top['score']:>9.3f}  "
              f"{top['doc_id']:<12} {kind}")

    # и три офтопик-вопроса не из golden - для полноты картины
    for question in ("Какая столица Франции?",
                     "Как приготовить борщ?",
                     "Кто выиграл чемпионат мира по футболу?"):
        top = retrieve(question, top_k=1)[0]
        print(f"{question[:50]:<52} {top['score']:>9.3f}  "
              f"{top['doc_id']:<12} офтопик")
else:
    print("[пропущено: Qdrant недоступен]")

In [ ]:
SCORE_THRESHOLD = 0.10   # стартовое значение по данным прогонов 3.1/3.3:
                         # офтопик ~0.05, худший «законный» вопрос 0.149.
                         # Подберите своё по таблице выше (упражнение 1).


def below_threshold(found: list[dict], threshold: float = SCORE_THRESHOLD) -> bool:
    """Рубеж 0: даже лучший чанк слишком далёк от вопроса."""
    return not found or found[0]["score"] < threshold


if QDRANT_AVAILABLE:
    for question in ("Какая столица Франции?",
                     "Сколько дней отпуска положено в году?"):
        found = retrieve(question)
        verdict = "ОТКАЗ до генерации" if below_threshold(found) else "пропустить дальше"
        print(f"[{found[0]['score']:.3f}] {question[:45]:<47} -> {verdict}")
else:
    print("[пропущено: Qdrant недоступен]")

Два свойства рубежа 0, за которые его любят в проде:

1. **Он бесплатный и мгновенный** — сравнение одного числа. На потоке вопросов, где заметная доля — офтопик и спам, порог экономит реальные деньги: LLM просто не вызывается.
2. **Его отказ — гарантированный**, а не «уговоренный»: модель невозможно уболтать промпт-инъекцией, потому что модели ещё нет.

И его принципиальный предел, который стоит повторить: порог ловит «*ничего похожего не нашлось*», но не «*нашлось не то*». «Упал прод» с его уверенными 0.341 у чужого документа пройдёт рубеж 0 всегда. Диагноз этого класса ошибок — только выше по конвейеру.

> ⚠️ Порог **не переносится** между эмбеддерами: у TF-IDF-косинуса и у e5 разные типичные диапазоны скоров (у нормированных нейроэмбеддингов почти всё живёт в узкой полосе 0.7–0.95). Сменили эмбеддер (урок 3.2, `EMB_BACKEND`) — пересчитайте распределения и выберите порог заново.

## 3. Рубеж 1 — промпт: краткое резюме

Промпт-оборону мы уже построили и *измерили* в уроке 3.3, поэтому здесь коротко. Три версии:

| версия | идея | что показали метрики |
|---|---|---|
| v1 «кратко и по делу» | базовые grounding-правила из 3.1 | отказы 3/3, но ответы куцые (cover 0.63–0.68) и случаются искажения |
| v2 «отвечай полно» | + «включая ВСЕ условия и цифры» | полнота выросла, **отказы просели** — подмена вопроса соседним |
| v3 «сначала реши» | решение об отказе отделено от стиля ответа | отказы 3/3, полнота на месте, «Упал прод» стал отказом |

Работающий принцип v3: **разделить решение и исполнение.** Пока «откажись, если нет ответа» и «отвечай полно» живут в одном правиле, они конкурируют — и побеждает то полнота, то дисциплина. v3 ставит решение об отказе *первым и отдельным* шагом. Мы используем его (`SYSTEM_RAG_V3` в ячейке настройки) как базу для всех следующих рубежей.

Но урок 3.3 же и показал предел: v3 — всё ещё уговор. «56 дней» случились при честном промпте и правильном контексте. Идём проверять ответ *снаружи*.

## 4. Рубеж 2 — цитаты, которые можно проверить

Требование «указывай источники [i]» из урока 3.1 — украшение, пока его никто не проверяет: модель может написать `[1]` под выдумкой (и наши прогоны такое видели: `«В базе знаний нет ответа на этот вопрос [1], [2], [3]»` — отказ с «источниками»!). Превращаем украшение в рубеж — двумя механическими проверками без единого вызова LLM:

1. **Валидность ссылок**: все `[i]` в ответе существуют (i не больше числа переданных чанков) и ответ-не-отказ содержит хотя бы одну ссылку. Регулярка.
2. **Дословная цитата (quote grounding)**: просим модель приложить к ответу точную цитату из контекста, на которую она опирается, — и проверяем **подстрокой**, что цитата действительно есть в контексте. Выдумать факт легко; выдумать дословную цитату, которая совпадёт с документом посимвольно, — практически невозможно. Это самый сильный дешёвый чек в уроке.

Для цитаты расширяем промпт отдельным блоком (формат — тэги, как учил урок 1.2 с разделителями):

In [ ]:
CITE_ADDON = (
    "\n5. Перед списком источников приведи ОДНУ дословную цитату из контекста "
    "(3-15 слов), на которой основан ответ, в формате:\n"
    "<quote>точная цитата из фрагмента</quote>"
)


def citations_valid(answer: str, n_chunks: int) -> bool:
    """Проверка 1: ссылки [i] существуют и есть хотя бы одна."""
    refs = [int(x) for x in re.findall(r"\[(\d+)\]", answer)]
    if not refs:
        return False
    return all(1 <= ref <= n_chunks for ref in refs)


def _normalize(text: str) -> str:
    """Сравниваем без регистра, пунктуации и лишних пробелов."""
    return re.sub(r"[^\wа-яё ]", "", text.lower(), flags=re.IGNORECASE).strip()


def quote_grounded(answer: str, found: list[dict]) -> bool | None:
    """Проверка 2: дословная цитата из <quote> реально есть в контексте.

    None - модель не дала цитату (тоже сигнал: считаем провалом мягко),
    True/False - цитата есть/нет в найденных чанках."""
    match = re.search(r"<quote>(.+?)</quote>", answer, re.DOTALL)
    if not match:
        return None
    quote = _normalize(match.group(1))
    context = _normalize(" ".join(c["text"] for c in found))
    return quote in context


# механику можно проверить и без LLM - на рукотворных примерах:
fake_found = [{"text": "Каждому сотруднику положено 28 календарных дней "
                       "оплачиваемого отпуска в год."}]
good = "Отпуск - 28 дней. <quote>28 календарных дней оплачиваемого отпуска</quote> [1]"
bad = "Отпуск - 45 дней. <quote>45 дней отпуска и 13-я зарплата</quote> [1]"
liar = "Отпуск - 45 дней. [1] [2] [3]"

print("честный ответ:  ссылки", citations_valid(good, 1),
      "| цитата", quote_grounded(good, fake_found))
print("выдумка с цитатой: ссылки", citations_valid(bad, 1),
      "| цитата", quote_grounded(bad, fake_found))
print("выдумка со ссылками на несуществующие чанки: ссылки",
      citations_valid(liar, 1))

Разберите три примера из ячейки: честный ответ проходит обе проверки; выдумка может *написать* тег `<quote>`, но её цитаты **нет в контексте** — `quote_grounded` вернёт `False`; а ссылки `[2]`, `[3]` при одном переданном чанке валит `citations_valid` без всякой семантики.

Пределы рубежа (честно): цитата подтверждает *опору*, а не *вывод* — модель может привести верную цитату «до 90 дней» и рядом написать «56» (наш мерцающий инцидент такую проверку прошёл бы наполовину: цитата верна, вывод — нет). Поэтому цитаты — фильтр грубых выдумок и обязательная основа для UX («показать, откуда ответ»), но не последний рубеж.

> ⚠️ Классическая ошибка интеграции: **фильтровать чанки после нумерации**. Если вы отбросили чанк №2 по порогу, а модель получила чанки с номерами [1], [3] — все её ссылки поедут. Нумеруйте *после* всех фильтраций.

## 5. Рубеж 3 — self-check: судья на страже, но по каскаду

Последний класс ошибок — «цитата верна, вывод искажён» — ловится только семантической проверкой: **подтверждается ли ответ контекстом?** Это ровно `supported()` из урока 3.3 (строгий судья, калибровка 8/8), применённый к целому ответу. Называется self-check, потому что модель проверяет сама себя — и тут обязаны прозвучать два предупреждения из 3.3:

1. Судья **унаследует слепые пятна** генератора (та же модель!). Наша калибровка показала, что бинарный вопрос «подтверждается? да/нет» qwen решает надёжно (8/8) — но это знание из *калибровки*, а не из веры. В проде self-check часто делают *другой* моделью.
2. Судья **стоит один вызов LLM на каждый ответ** — почти удвоение цены. Поэтому каскад: бесплатные рубежи 0 и 2 — всегда, self-check — только для ответов, прошедших их (а при жёстком бюджете — только для подозрительных: без цитаты, с низким скором, длинных).

Политика при провале: **один retry** — генерируем заново с лёгкой температурой (при t=0 вторая попытка повторит первую дословно; небольшая температура даёт «мерцающим» сбоям вроде «56 дней» шанс исчезнуть), после второго провала — честный отказ. Ограничение в один retry принципиально: бесконечный цикл «генерируй, пока судья не одобрит» — это уже не защита, а подгонка ответа под судью.

In [ ]:
VERIFY_SYSTEM = (      # строгий судья из урока 3.3, калибровка 8/8
    "Ты проверяешь факты. Дан фрагмент документа и утверждение. Ответь одним "
    "словом: «да» — если утверждение прямо подтверждается фрагментом, «нет» — "
    "если не подтверждается или противоречит."
)


def self_check(answer: str, found: list[dict]) -> bool:
    """Рубеж 3: ответ целиком подтверждён найденным контекстом?"""
    context = "\n".join(c["text"] for c in found)
    clean = re.sub(r"<quote>.+?</quote>|\[\d+\]", "", answer, flags=re.DOTALL)
    raw = chat([{"role": "system", "content": VERIFY_SYSTEM},
                {"role": "user", "content":
                 f"Фрагмент:\n{context}\n\nУтверждение: {clean.strip()}"}],
               num_predict=10)
    return raw.strip().lower().startswith("да")


if QDRANT_AVAILABLE and OLLAMA_AVAILABLE:
    found = retrieve("Сколько дней отпуска положено в году?")
    honest = "Каждому сотруднику положено 28 календарных дней отпуска в год."
    fabricated = "Сотрудникам положено 45 дней отпуска и 13-я зарплата."
    print("честный ответ    ->", "прошёл" if self_check(honest, found) else "ПРОВАЛ")
    print("выдумка          ->", "прошёл" if self_check(fabricated, found) else "ПРОВАЛ")
else:
    print("[пропущено: нужны Qdrant и Ollama]")

## 6. Рубеж 4 — отказ как фича

Отказ в плохо спроектированной системе выглядит как ошибка — и команды «чинят» его, ослабляя защиту (мы видели, как v2 разменял отказы на полноту). В хорошо спроектированной системе отказ — это **продуктовый ответ**: честный, единообразный и полезный.

Три инженерных решения:

1. **Канонический текст отказа — константа приложения**, а не творчество модели. Что бы модель ни написала, наружу уходит `REFUSAL_TEXT` — одинаковый, переводимый, тестируемый. Заодно исчезает проблема «отказался другими словами».
2. **Детектор отказа — по паттернам**, а не по одной фразе (урок 3.3, ловушка `parking`): «нет ответа», «нет информации», «не нашлось», «не могу ответить» — все считаются отказом модели.
3. **Отказ приходит с подсказкой**: раз ретривер уже отработал, покажем пользователю, *какие темы есть рядом* — «ответа нет, но в базе есть про: больничные, отпуск». Стоит ноль (данные уже в руках), а UX меняет радикально: тупик превращается в навигацию.

In [ ]:
REFUSAL_PATTERNS = ("нет ответа", "нет информации", "не нашлось",
                    "не могу ответить", "нет данных")


def is_refusal(answer: str) -> bool:
    """Детектор отказа по паттернам (урок 3.3, случай parking)."""
    lowered = answer.lower()
    return any(pattern in lowered for pattern in REFUSAL_PATTERNS)


def refusal_response(found: list[dict]) -> dict:
    """Канонический отказ + подсказка ближайших тем из уже найденных чанков."""
    topics = []
    for chunk in found:
        title = chunk.get("title", "")
        if title and title not in topics:
            topics.append(title)
    hint = f" Ближайшие темы в базе: {', '.join(topics[:3])}." if topics else ""
    return {"answer": REFUSAL_TEXT + "." + hint, "sources": [], "verdict": "refused"}


# детектор проверяется без LLM:
for text in ("В базе знаний нет ответа на этот вопрос [1].",
             "В базе знаний нет информации о наличии парковки.",
             "Отпуск - 28 календарных дней. [1]"):
    print(f"{'отказ ' if is_refusal(text) else 'ответ '} <- «{text[:55]}»")

## 7. Сборка: guarded_rag_answer

Собираем рубежи в один конвейер. Требования к сборке:

- каждый рубеж возвращает свой **вердикт** — итоговый ответ несёт не только текст, но и «паспорт проверок» (что сработало, что нет): это пища для логов и метрик урока 3.7;
- порядок — по стоимости: **порог → генерация → детектор отказа → цитаты → self-check**;
- при провале цитат или self-check — **один retry**, затем канонический отказ: лучше честное «не знаю», чем непроверенный ответ.

In [ ]:
def guarded_rag_answer(question: str, top_k: int = 3,
                       threshold: float = SCORE_THRESHOLD,
                       max_retries: int = 1) -> dict:
    """RAG с эшелонированной обороной. Возвращает ответ + паспорт проверок."""
    checks = {}

    # --- Рубеж 0: порог релевантности (LLM ещё не вызывалась) ---------------
    found = retrieve(question, top_k)
    checks["top_score"] = round(found[0]["score"], 3) if found else 0.0
    if below_threshold(found, threshold):
        checks["threshold"] = "REFUSED"
        return {**refusal_response([]), "verdict": "refused_by_threshold",
                "checks": checks, "llm_calls": 0}
    checks["threshold"] = "ok"

    context = "\n\n".join(f"[{i}] {c['title']}:\n{c['text']}"
                          for i, c in enumerate(found, 1))
    messages = [{"role": "system", "content": SYSTEM_RAG_V3 + CITE_ADDON},
                {"role": "user", "content":
                 f"<context>\n{context}\n</context>\n\nВопрос: {question}"}]

    llm_calls = 0
    for attempt in range(1 + max_retries):
        # retry - с лёгкой температурой: при t=0 вторая попытка почти наверняка
        # повторит первую дословно, и смысла в ней не будет
        answer = chat(messages, temperature=0.0 if attempt == 0 else 0.3)
        llm_calls += 1                                # Рубеж 1: промпт v3

        if is_refusal(answer):                        # модель отказалась сама
            checks["model_refused"] = True
            return {**refusal_response(found), "verdict": "refused_by_model",
                    "checks": checks, "llm_calls": llm_calls}

        # --- Рубеж 2: механика цитат (бесплатно) ----------------------------
        checks["citations"] = citations_valid(answer, len(found))
        checks["quote"] = quote_grounded(answer, found)
        if not checks["citations"] or checks["quote"] is False:
            checks[f"attempt_{attempt + 1}"] = "citation_fail"
            continue                                  # retry

        # --- Рубеж 3: self-check (ещё один вызов LLM) -----------------------
        checks["self_check"] = self_check(answer, found)
        llm_calls += 1
        if checks["self_check"]:
            clean = re.sub(r"<quote>.+?</quote>", "", answer,
                           flags=re.DOTALL).strip()
            return {"answer": clean, "sources": found,
                    "verdict": "ok", "checks": checks, "llm_calls": llm_calls}
        checks[f"attempt_{attempt + 1}"] = "self_check_fail"

    # обе попытки провалили проверки - честный отказ
    return {**refusal_response(found), "verdict": "refused_after_checks",
            "checks": checks, "llm_calls": llm_calls}


print("конвейер собран: retrieve -> порог -> v3+цитата -> отказ? -> "
      "цитаты -> self-check -> retry x1 -> ответ/отказ")

In [ ]:
if QDRANT_AVAILABLE and OLLAMA_AVAILABLE:
    SCENARIOS = [
        ("обычный вопрос",    "Сколько дней отпуска положено в году?"),
        ("чистый офтопик",    "Какая столица Франции?"),
        ("близкая ловушка",   "Сколько платят за переработки в выходные?"),
        ("уверенный мусор",   "Упал прод, что делать в первую очередь?"),
    ]
    for label, question in SCENARIOS:
        result = guarded_rag_answer(question)
        print(f"=== {label}: {question}")
        print(f"    вердикт: {result['verdict']} "
              f"(вызовов LLM: {result['llm_calls']})")
        print(f"    проверки: {result['checks']}")
        print(f"    ответ: «{result['answer'][:70]}»\n")
else:
    print("[пропущено: нужны Qdrant и Ollama]")

Что ожидать от четырёх сценариев (проверьте на своей машине):

- **обычный вопрос** — вердикт `ok`, две галочки-проверки, 2 вызова LLM (генерация + self-check);
- **чистый офтопик** — `refused_by_threshold`, **0 вызовов LLM**: скор ~0.05 не дотягивает до порога;
- **близкая ловушка** — порог она *пройдёт* (тема-то рядом), и дальше исход зависит от промпта v3 и self-check: в прогонах 3.3 v3 на ней отказывался; если модель всё же ответит про «ночные вызовы» — self-check с вопросом должен поймать несоответствие;
- **уверенный мусор** («Упал прод», скор 0.341 у чужого документа) — порог бессилен (рубеж 0 пройден), но v3 в наших прогонах отказывался сам, а если не откажется — ответ из весов не найдёт дословной цитаты в контексте и провалит рубеж 2.

Заметьте, как рубежи **страхуют друг друга**: у каждого инцидента из карты дыр есть минимум два шанса быть пойманным.

И последний штрих — мини-eval: дисциплина урока 3.3 никуда не девается. Прогоним весь golden через защищённый конвейер и сравним отказы с незащищённым v3:

In [ ]:
if QDRANT_AVAILABLE and OLLAMA_AVAILABLE:
    stats = {"ok": 0, "refused_by_threshold": 0, "refused_by_model": 0,
             "refused_after_checks": 0, "refused": 0}
    false_refusals = true_refusals = total_llm_calls = 0

    for item in golden:
        result = guarded_rag_answer(item["question"])
        stats[result["verdict"]] = stats.get(result["verdict"], 0) + 1
        total_llm_calls += result["llm_calls"]
        refused = result["verdict"] != "ok"
        if item["answerable"] and refused:
            false_refusals += 1
            print(f"  ложный отказ: {item['id']} ({result['verdict']})")
        if not item["answerable"] and refused:
            true_refusals += 1

    n_traps = sum(not g["answerable"] for g in golden)
    n_answerable = len(golden) - n_traps
    print(f"\nвердикты: {stats}")
    print(f"верные отказы:  {true_refusals}/{n_traps}")
    print(f"ложные отказы:  {false_refusals}/{n_answerable} "
          f"(цена защиты - следите за этим числом!)")
    print(f"вызовов LLM на {len(golden)} вопросов: {total_llm_calls} "
          f"(без каскада было бы {len(golden) * 2}+)")
else:
    print("[пропущено: нужны Qdrant и Ollama]")

У защиты есть цена, и мини-eval её показывает: **ложные отказы**. Каждый рубеж, который ловит галлюцинации, иногда ловит и честные ответы (self-check строг — калибровка 3.3 показала ошибки именно в сторону строгости). Это тот же trade-off, что у порога: безопасность против полезности. Правильная точка баланса зависит от продукта: для внутренней базы знаний ложный отказ — мелкое неудобство, для медицинского ассистента ошибочный *ответ* — катастрофа, и защиту затягивают сильнее.

## ⚠️ Частые ошибки

| ошибка | чем оборачивается | лекарство |
|---|---|---|
| порог «из головы» или с чужого проекта | либо режет честные вопросы, либо пропускает всё | распределения скоров по своему golden (ячейка раздела 2) |
| один порог после смены эмбеддера | у TF-IDF и e5 разные диапазоны скоров — старый порог бессмыслен | пересчитать распределения, порог хранить рядом с именем коллекции |
| верить `[1]` как факту | модель ставит ссылки и под выдумкой, и под отказом | проверять валидность ссылок + дословную цитату подстрокой |
| фильтровать чанки после нумерации | все `[i]` в ответе съезжают | нумеровать после всех фильтраций |
| self-check на каждый ответ без каскада | почти удвоение цены и латентности | дешёвые проверки первыми; судья — последним и не всегда |
| retry до победного | «генерируй, пока судья не одобрит» = подгонка под судью | max_retries=1, дальше честный отказ |
| «чинить» участившиеся отказы ослаблением защиты | v2 из урока 3.3: полнота выросла, ловушки провалились | сначала диагноз по вердиктам: какой рубеж отказывает и почему |
| считать порог защитой от «уверенного мусора» | скор меряет похожесть слов, а не правильность документа | рубежи 2–3 + канарейка в golden |
| отказ без альтернативы | пользователь упирается в стену и уходит | подсказка ближайших тем — бесплатно, данные уже в руках |

## 🏋️ Упражнения

**1. Подберите порог по данным.** Ячейка раздела 2 печатает top-скоры всех вопросов golden и трёх офтопиков. Постройте по ним два списка (отвечаемые / ловушки+офтопик), найдите порог с нулём ложных отказов и порог с максимумом пойманного мусора. Совпадают ли они? Что вы выберете для «Векторики» и что — для медицинского бота?

**2. Сломайте валидатор цитат.** Придумайте ответ, который пройдёт `citations_valid` и `quote_grounded`, но при этом искажает контекст (подсказка: верная цитата + неверный вывод рядом — наш «56 дней»-паттерн). Какой рубеж обязан его поймать? Проверьте, ловит ли.

**3. Посчитайте цену обороны.** Для потока из 100 вопросов, где 20 — офтопик ниже порога, 10 — отказы модели, а 5 — retry по провалу проверок, посчитайте: сколько вызовов LLM сделает наш каскад и сколько сделала бы наивная схема «self-check на всё, без порога». Напишите функцию `estimate_cost(n, offtopic_rate, refusal_rate, retry_rate)`.

## 📝 Мини-квиз

<details><summary><b>1. Почему порог релевантности не поймал «Упал прод», хотя это самый проблемный вопрос базы?</b></summary>

Потому что порог ловит «ничего похожего не нашлось», а тут ретривер *уверенно нашёл не то*: лексическое пересечение с чужим документом дало скор 0.341 — выше любого разумного порога. Скор меряет похожесть слов, а не правильность документа. Этот класс ошибок ловят рубежи выше: промпт v3 («прямой ответ именно на этот вопрос»), проверка цитат и self-check.
</details>

<details><summary><b>2. Чем дословная цитата в &lt;quote&gt; сильнее, чем ссылка [1]?</b></summary>

Ссылку модель ставит «из вежливости» — она есть и под выдумками, и под отказами (мы видели «нет ответа [1], [2], [3]»). Дословную цитату нельзя выдумать безнаказанно: она проверяется подстрокой в контексте — без LLM, за микросекунды. Но цитата подтверждает опору, а не вывод: верная цитата рядом с искажённым выводом проходит — для этого есть self-check.
</details>

<details><summary><b>3. Self-check делает та же модель, что и генерацию. Это вообще законно?</b></summary>

Законно, если доверие судье *измерено*: наша калибровка 3.3 показала, что бинарный вопрос «подтверждается ли утверждение фрагментом?» qwen2.5:3b решает надёжно (8/8) — задача проверки проще задачи генерации. Но слепые пятна общие, поэтому в проде self-check часто делают другой моделью, а калибровка обязательна в любом случае.
</details>

<details><summary><b>4. Зачем ограничивать retry одной попыткой?</b></summary>

Бесконечный цикл «генерируй, пока судья не одобрит» — это не защита, а подгонка ответа под судью: рано или поздно модель случайно сформулирует то, что судья пропустит (в том числе ошибочное — судья не идеален). Один retry снимает «мерцающие» сбои (как «56 дней»), дальше — честный отказ.
</details>

<details><summary><b>5. После включения защиты доля отказов выросла, пользователи жалуются. Первый шаг?</b></summary>

Диагноз по вердиктам, а не ослабление защиты: конвейер возвращает паспорт проверок — смотрим, *какой* рубеж отказывает. Порог? — пересмотреть по распределениям. Self-check? — проверить его калибровку и ложные отказы на golden. Модель сама? — смотреть промпт. «Чинить отказы» ослаблением защиты — путь v2 из урока 3.3: полнота вырастет, ловушки провалятся.
</details>

## Решения

**Решение 1 — порог по данным.** Код собирает распределения и перебирает пороги-кандидаты; интерпретация зависит от ваших чисел, ориентиры — в комментариях.

In [ ]:
if QDRANT_AVAILABLE:
    answerable_scores, trap_scores = [], []
    for item in golden:
        score = retrieve(item["question"], top_k=1)[0]["score"]
        (answerable_scores if item["answerable"] else trap_scores).append(score)
    for question in ("Какая столица Франции?", "Как приготовить борщ?",
                     "Кто выиграл чемпионат мира по футболу?"):
        trap_scores.append(retrieve(question, top_k=1)[0]["score"])

    print(f"отвечаемые:      min={min(answerable_scores):.3f} "
          f"max={max(answerable_scores):.3f}")
    print(f"ловушки+офтопик: min={min(trap_scores):.3f} "
          f"max={max(trap_scores):.3f}")

    print(f"\n{'порог':>6} {'ложных отказов':>15} {'мусора поймано':>15}")
    for threshold in (0.05, 0.08, 0.10, 0.12, 0.15, 0.20):
        false_refused = sum(s < threshold for s in answerable_scores)
        caught = sum(s < threshold for s in trap_scores)
        print(f"{threshold:>6.2f} {false_refused:>10}/{len(answerable_scores)} "
              f"{caught:>13}/{len(trap_scores)}")
    # Ориентиры: для базы знаний берут порог с 0 ложных отказов (полезность),
    # для рискованных доменов - сдвигают вправо, принимая ложные отказы.
    # Близкие ловушки («переработки») почти наверняка выше любого порога -
    # их дело решают рубежи 1-3, порог отвечает только за офтопик.
else:
    print("[пропущено: Qdrant недоступен]")

**Решение 2 — обход валидатора цитат.** Атака: верная цитата + искажённый вывод рядом. Обе механические проверки проходят (ссылки валидны, цитата дословная) — ловить обязан self-check, потому что только он сравнивает *смысл ответа* с контекстом.

In [ ]:
attack = ("Из-за границы можно работать 56 дней в году. "
          "<quote>разрешена до 90 дней в году</quote> [1]")
attack_found = [{"text": "Работа из-за границы разрешена до 90 дней в году; "
                         "обязательно уведомить HR минимум за 2 недели.",
                 "title": "Удалённая работа"}]

print("citations_valid:", citations_valid(attack, 1))       # True - пройдено
print("quote_grounded: ", quote_grounded(attack, attack_found))  # True - пройдено!

if OLLAMA_AVAILABLE:
    verdict = self_check(attack, attack_found)
    print("self_check:     ", verdict, "<- последний рубеж обязан сказать False")
else:
    print("self_check:      [пропущено: Ollama недоступна] - "
          "проверьте на своей машине: ожидается False")

**Решение 3 — цена обороны.** Считаем вызовы LLM. Наивная схема: каждый вопрос = генерация + self-check. Каскад: офтопик не доходит до LLM вовсе, отказы модели не проверяются судьёй, retry добавляет генерацию и оба чека.

In [ ]:
def estimate_cost(n: int, offtopic_rate: float, refusal_rate: float,
                  retry_rate: float) -> dict:
    """Вызовы LLM: каскад против наивной схемы «self-check на всё»."""
    offtopic = n * offtopic_rate                # 0 вызовов: порог
    refusals = n * refusal_rate                 # 1 вызов: генерация без судьи
    retried = n * retry_rate                    # 2 генерации + 2 self-check
    normal = n - offtopic - refusals - retried  # генерация + self-check
    cascade = refusals * 1 + retried * 4 + normal * 2
    naive = n * 2                               # порога нет: все по 2 вызова
    return {"каскад": round(cascade), "наивная схема": round(naive),
            "экономия": f"{(1 - cascade / naive):.0%}"}


print(estimate_cost(100, offtopic_rate=0.20, refusal_rate=0.10, retry_rate=0.05))
# offtopic 20 вопросов бесплатно + 10 отказов по 1 вызову = главные источники
# экономии; retry дорог (4 вызова), но редок. Чем грязнее поток - тем больше
# экономит порог: защита и бюджет тут на одной стороне.

## Итоги

- **Промпт — уговор, оборона — система.** Четыре задокументированных инцидента из уроков 3.1–3.3 разобраны на классы, и у каждого класса — свой рубеж: порог (офтопик), промпт v3 (подмена вопроса), проверяемые цитаты (грубые выдумки), self-check (искажение контекста), канонический отказ (хрупкий детектор).
- **Дешёвое — первым.** Порог — сравнение числа (0 вызовов LLM), цитаты — регулярка и подстрока, self-check — последним и не всегда. Каскад экономит на грязном потоке десятки процентов вызовов.
- **Порог принципиально не ловит «уверенный мусор»**: скор меряет похожесть слов. «Упал прод» с 0.341 у чужого документа проходит любой порог — его ловят верхние рубежи.
- **Дословная цитата — сильнейший дешёвый чек**: выдумать её нельзя, проверить — подстрока. Но она подтверждает опору, не вывод — «верная цитата, искажённый вывод» ловится только self-check'ом.
- **У защиты есть цена — ложные отказы.** Следите за ними в мини-eval так же, как за пойманными галлюцинациями; баланс — продуктовое решение, а не константа.
- **Retry — один.** Дальше — честный отказ с подсказкой ближайших тем: отказ — это фича, а не ошибка.

| приём урока | родом из |
|---|---|
| скоры ретривера, «одна коллекция — одна модель» | уроки 2.4, 3.2 |
| grounding-промпт, разделители, право не знать | уроки 1.2, 3.1 |
| промпт v3, судья supported(), калибровка, ловушки golden | урок 3.3 |
| каскад «дешёвое первым» | тот же принцип, что recall→precision в уроке 2.5 |

**Что дальше.** Наш RAG умеет отвечать, отказываться и проверять сам себя — но он всё ещё *пассивен*: один запрос — один поиск — один ответ. Урок 3.5 делает следующий шаг: **агенты** — LLM, которая сама решает, что сделать: поискать ещё раз с другой формулировкой, заглянуть в другой источник, вызвать инструмент. Цикл «думай → действуй → смотри на результат» на чистом Python + Ollama, без фреймворков.